# 03 Sequence-to-Sequence Models | نماذج تسلسل إلى تسلسل

## 📚 Learning Objectives | أهداف التعلم

By completing this notebook (~20 min), you will:
- Build a simple **encoder–decoder** (seq2seq) model with LSTM
- Use it for a **toy task** (e.g. reverse sequence or short copy) and see input → output
- Understand why we use encoder–decoder instead of a single RNN for translation/summarization

---

## 🌍 Real life | في الواقع

**Where is this used?** Seq2seq is used in **machine translation**, **summarization**, **dialogue**, and **speech-to-text**.

**In this notebook we use** an **encoder–decoder** (encoder RNN → context → decoder RNN) to map one sequence to another. We use **seq2seq** (instead of one RNN that reads and writes in one pass) **because** input and output can have **different lengths**; the encoder summarizes the input into a context, and the decoder generates the output step by step.

---

**Before starting:** Run the imports cell below.

## Theory (short) | النظرية

- **Encoder:** Reads the input sequence and produces a **context vector** (e.g. last hidden state or attention over encoder outputs).
- **Decoder:** Starts from the context (and optionally a start token) and generates the output sequence step by step.
- **Seq2seq:** Input length can differ from output length; encoder summarizes, decoder generates.
- **We use encoder–decoder** instead of a single RNN when input and output lengths differ (e.g. translation) or when we need a clear "read then write" structure.

## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** TensorFlow/Keras, NumPy. We use a **toy task**: fixed-length integer sequences (e.g. copy or reverse) so we don't need a translation dataset.

**Dataset:** Synthetic — toy integer sequences (no download; used to demonstrate encoder–decoder).

**Outputs:** Model structure, one example input → output (or loss over a few steps), and a short explanation.

## Step 1: Imports and create toy sequences (we use a simple copy task so seq2seq runs in ~5 min)

In [ ]:
import numpy as np

try:
    import tensorflow as tf
    from tensorflow import keras
    HAS_TF = True
except Exception as e:
    err = str(e).lower()
    if "charset_normalizer" in err or "md__mypyc" in err or "partially initialized" in err:
        print("⚠️ Fix: pip install --upgrade charset-normalizer requests, then restart kernel.")
        raise RuntimeError("Fix: pip install --upgrade charset-normalizer requests, then restart kernel.") from e
    HAS_TF = False

if HAS_TF:
    seq_len = 5
    vocab_size = 10
    n_samples = 200
    np.random.seed(42)
    x = np.random.randint(1, vocab_size, (n_samples, seq_len))
    y = x[:, ::-1]
    print("Input shape:", x.shape, "Output shape:", y.shape)
    print("Example: input", x[0].tolist(), "-> target (reversed)", y[0].tolist())
else:
    print("Install TensorFlow: pip install tensorflow")

## Step 2: Build encoder–decoder (we use encoder–decoder instead of one RNN because input and output are separate sequences)

In [ ]:
if HAS_TF:
    encoder_inp = keras.Input(shape=(seq_len,))
    emb = keras.layers.Embedding(vocab_size, 32)(encoder_inp)
    _, state_h, state_c = keras.layers.LSTM(32, return_state=True)(emb)
    encoder_states = [state_h, state_c]

    decoder_inp = keras.Input(shape=(seq_len,))
    emb_dec = keras.layers.Embedding(vocab_size, 32)(decoder_inp)
    decoder_lstm = keras.layers.LSTM(32, return_sequences=True, return_state=True)
    dec_out, _, _ = decoder_lstm(emb_dec, initial_state=encoder_states)
    out = keras.layers.Dense(vocab_size, activation="softmax")(dec_out)

    model = keras.Model([encoder_inp, decoder_inp], out)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.summary()

## Step 3: Prepare decoder input (shift right: [START] + target[:-1]) and train (2 epochs)

In [ ]:
if HAS_TF:
    dec_inp = np.zeros_like(y)
    dec_inp[:, 1:] = y[:, :-1]
    dec_inp[:, 0] = 0
    y_one_step = y
    history = model.fit([x, dec_inp], y_one_step, epochs=2, batch_size=32, verbose=1)
    pred = model.predict([x[:3], dec_inp[:3]], verbose=0)
    pred_class = np.argmax(pred, axis=-1)
    print("Sample predictions (target vs pred):")
    for i in range(3):
        print("  target:", y[i].tolist(), "pred:", pred_class[i].tolist())

## 🧩 Mini-exercise | تمرين مصغر

**Try it:** Change the sequence length (e.g. 5 → 7) in the toy data and retrain. Does the model still learn to reverse? Or try a different task (e.g. copy the sequence instead of reverse).

---

## ✅ Summary | الملخص

**What you did:** Built an encoder–decoder (LSTM) for a toy reverse-sequence task; trained for 2 epochs and showed sample input → output.

**In real life you'd also:** Use attention over encoder outputs, real translation data, and teacher forcing / scheduled sampling.

**The main idea:** Seq2seq = encoder (summarize input) + decoder (generate output); we use it when input and output lengths differ (e.g. translation).

**Next:** `04_transformer_attention` introduces attention; `05_bert_finetuning` fine-tunes an encoder for classification.